In [ ]:
{
    "cells": [
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "# ADGroupReview.ipynb\n",
                "**Purpose**: Review members and ACLs for a specific AD group and tag decisions for CAB post-analysis.\n",
                "\n",
                "This notebook supports:\n",
                "- Manual or rule-based tagging\n",
                "- CSV + Markdown export for CAB reports\n",
                "- Offline review with no pip installs required"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": null,
            "outputs": [],
            "source": [
                "# Setup\n",
                "import json\n",
                "import pandas as pd\n",
                "from pathlib import Path\n",
                "\n",
                "# Set target group\n",
                "group_name = input(\"Enter group name (e.g. Finance-UK): \")\n",
                "export_dir = Path(f\"../exports/{group_name}\")\n",
                "\n",
                "# Locate latest snapshot\n",
                "json_file = sorted(export_dir.glob(\"group_snapshot_*.json\"), reverse=True)[0]\n",
                "with open(json_file, \"r\", encoding=\"utf-8\") as f:\n",
                "    group_data = json.load(f)"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": null,
            "outputs": [],
            "source": [
                "# Extract Members + ACLs\n",
                "members = group_data.get(\"Members\", [])\n",
                "acls = group_data.get(\"ACLs\", [])\n",
                "\n",
                "df_members = pd.DataFrame(members, columns=[\"Member\"])\n",
                "df_members[\"Decision\"] = \"\"\n",
                "\n",
                "df_acls = pd.DataFrame(acls)\n",
                "if not df_acls.empty:\n",
                "    df_acls[\"Decision\"] = \"\""
            ]
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": null,
            "outputs": [],
            "source": [
                "# Preview Tables\n",
                "df_members.style.set_properties(**{'text-align': 'left'})"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": null,
            "outputs": [],
            "source": [
                "# Preview ACLs (if any)\n",
                "df_acls.style.set_properties(**{'text-align': 'left'})"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": null,
            "outputs": [],
            "source": [
                "# Optional: apply rule-based tagging\n",
                "df_members.loc[df_members[\"Member\"].str.contains(\"admin\", case=False, na=False), \"Decision\"] = \"CAB Review\"\n",
                "df_acls.loc[df_acls[\"Rights\"].str.contains(\"FullControl\", case=False, na=False), \"Decision\"] = \"CAB Review\""
            ]
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": null,
            "outputs": [],
            "source": [
                "# Export CSVs\n",
                "df_members.to_csv(export_dir / \"review_members.csv\", index=False)\n",
                "df_acls.to_csv(export_dir / \"review_acls.csv\", index=False)\n",
                "print(\" CSV exports complete\")"
            ]
        },
        {
            "cell_type": "code",
            "metadata": {},
            "execution_count": null,
            "outputs": [],
            "source": [
                "# Export Markdown Summary\n",
                "lines = [f\"# AD Group Review: {group_name}\\n\"]\n",
                "\n",
                "lines.append(\"## Members\\n\")\n",
                "for _, row in df_members.iterrows():\n",
                "    lines.append(f\"- **{row['Member']}**: `{row['Decision']}`\")\n",
                "\n",
                "if not df_acls.empty:\n",
                "    lines.append(\"\\n## ACL Entries\\n\")\n",
                "    for _, row in df_acls.iterrows():\n",
                "        lines.append(f\"- **{row['Identity']} / {row['Rights']}**: `{row['Decision']}`\")\n",
                "\n",
                "with open(export_dir / \"review_summary.md\", \"w\", encoding=\"utf-8\") as f:\n",
                "    f.write(\"\\n\".join(lines))\n",
                "print(\" Markdown summary exported\")"
            ]
        },
        {
            "cell_type": "markdown",
            "metadata": {},
            "source": [
                "🎯 Use `review_members.csv` and `review_summary.md` for submission to your CAB board."
            ]
        }
    ],
    "metadata": {
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        },
        "language_info": {
            "name": "python",
            "version": "3.11"
        }
    },
    "nbformat": 4,
    "nbformat_minor": 5
}